# Accident & Fire detection

## ✅ Libraries & Environment

In [2]:
!pip install -qU roboflow ultralytics wandb

In [3]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
roboflow_api_key = user_secrets.get_secret("RoboFlow")
wandb_api_key = user_secrets.get_secret("WandB_SafeSpace")

wandb.login(key=wandb_api_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ahmed-hossam (ahmed-hossam-suez-canal-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## ❌ Downloading dataset from Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=roboflow_api_key)
project = rf.workspace("ws01-so2ko").project("fire-detaction")
version = project.version(3)
dataset = version.download("yolo26")

## ❌ Check Dataset Imbalance

In [ ]:
import os

labels_dir = "/kaggle/working/Fire-Detaction-3/train/labels"

class_counts = {}

for file in os.listdir(labels_dir):
    file_path = os.path.join(labels_dir, file)
    
    with open(file_path, "r") as f:
        lines = f.readlines()
        for line in lines:
            class_id = line.split()[0]
            class_counts[class_id] = class_counts.get(class_id, 0) + 1

print("Class counts:", class_counts)

max_class = max(class_counts.values())
min_class = min(class_counts.values())

print("Imbalance ratio:", max_class / min_class)

image_classes = {}

for file in os.listdir(labels_dir):
    file_path = os.path.join(labels_dir, file)
    
    with open(file_path, "r") as f:
        lines = f.readlines()
        classes_in_image = set(line.split()[0] for line in lines)
        
        for cls in classes_in_image:
            image_classes[cls] = image_classes.get(cls, 0) + 1

print("Image-level counts:", image_classes)

In [ ]:
!echo "Train: $(ls -1 /kaggle/working/Fire-Detaction-3/train/images | wc -l)" && echo "Valid: $(ls -1 /kaggle/working/Fire-Detaction-3/valid/images | wc -l)" && echo "Test:  $(ls -1 /kaggle/working/Fire-Detaction-3/test/images | wc -l)"

## ✅ Merging 2 datasets from kaggle

In [4]:
import os
import shutil

# ── Paste your exact paths here ──
OUTPUT_PATH = "/kaggle/working/v2_acc_and_fire_datasets_merged"

ACCIDENT_TRAIN_IMAGES = "/kaggle/input/datasets/maryamsamirelsayed/accident-types-and-detection-dataset/train/images"   # change
ACCIDENT_TRAIN_LABELS = "/kaggle/input/datasets/maryamsamirelsayed/accident-types-and-detection-dataset/train/labels"   # change
ACCIDENT_VAL_IMAGES   = "/kaggle/input/datasets/maryamsamirelsayed/accident-types-and-detection-dataset/valid/images"   # change
ACCIDENT_VAL_LABELS   = "/kaggle/input/datasets/maryamsamirelsayed/accident-types-and-detection-dataset/valid/labels"   # change
ACCIDENT_TEST_IMAGES  = "/kaggle/input/datasets/maryamsamirelsayed/accident-types-and-detection-dataset/test/images"    # change
ACCIDENT_TEST_LABELS  = "/kaggle/input/datasets/maryamsamirelsayed/accident-types-and-detection-dataset/test/labels"    # change

FIRE_TRAIN_IMAGES = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/train/images"  # change
FIRE_TRAIN_LABELS = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/train/labels"  # change
FIRE_VAL_IMAGES   = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images"    # change
FIRE_VAL_LABELS   = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels"    # change
FIRE_TEST_IMAGES  = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/images"   # change
FIRE_TEST_LABELS  = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/labels"   # change

# ── Merged class list ──
CLASSES = [
    'bus', 'bus_bus_accident', 'bus_object_accident', 'bus_person_accident',
    'bus_truck_accident', 'car', 'car_bus_accident', 'car_car_accident',
    'car_motorcycle_accident', 'car_object_accident', 'car_person_accident',
    'car_truck_accident', 'motorcycle', 'motorcycle_bus_accident',
    'motorcycle_motorcycle_accident', 'motorcycle_object_accident',
    'motorcycle_person_accident', 'motorcycle_truck_accident', 'truck',
    'truck_object_accident', 'truck_person_accident', 'truck_truck_accident',
    'smoke', 'fire'
]

# ── Create output folders ──
for split in ['train', 'val', 'test']:
    os.makedirs(f"{OUTPUT_PATH}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUTPUT_PATH}/labels/{split}", exist_ok=True)

# ── Copy function ──
def copy_split(img_dir, lbl_dir, prefix, class_offset, split):
    if not os.path.exists(img_dir):
        print(f"  Skipping {split} — folder not found: {img_dir}")
        return
    count = 0
    for fname in os.listdir(img_dir):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        stem = os.path.splitext(fname)[0]
        shutil.copy(f"{img_dir}/{fname}", f"{OUTPUT_PATH}/images/{split}/{prefix}_{fname}")
        lbl_src = f"{lbl_dir}/{stem}.txt"
        lbl_dst = f"{OUTPUT_PATH}/labels/{split}/{prefix}_{stem}.txt"
        if os.path.exists(lbl_src):
            with open(lbl_src) as f:
                lines = f.readlines()
            with open(lbl_dst, 'w') as f:
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        parts[0] = str(int(parts[0]) + class_offset)
                        f.write(' '.join(parts) + '\n')
        count += 1
    print(f"  [{split}] {prefix}: {count} images copied")

# ── Run merge ──
print("Copying accident dataset...")
copy_split(ACCIDENT_TRAIN_IMAGES, ACCIDENT_TRAIN_LABELS, 'acc', class_offset=0,  split='train')
copy_split(ACCIDENT_VAL_IMAGES,   ACCIDENT_VAL_LABELS,   'acc', class_offset=0,  split='val')
copy_split(ACCIDENT_TEST_IMAGES,  ACCIDENT_TEST_LABELS,  'acc', class_offset=0,  split='test')

print("Copying fire dataset...")
copy_split(FIRE_TRAIN_IMAGES, FIRE_TRAIN_LABELS, 'fire', class_offset=22, split='train')
copy_split(FIRE_VAL_IMAGES,   FIRE_VAL_LABELS,   'fire', class_offset=22, split='val')
copy_split(FIRE_TEST_IMAGES,  FIRE_TEST_LABELS,  'fire', class_offset=22, split='test')

# ── Save data.yaml ──
with open(f"{OUTPUT_PATH}/data.yaml", 'w') as f:
    f.write(f"path: {OUTPUT_PATH}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("test: images/test\n")
    f.write(f"nc: {len(CLASSES)}\n")
    f.write(f"names: {CLASSES}\n")

# ── Summary ──
print("\n✅ Merge complete!")
for split in ['train', 'val', 'test']:
    n = len(os.listdir(f"{OUTPUT_PATH}/images/{split}"))
    print(f"  {split}: {n} images")
print(f"  Total classes: {len(CLASSES)}")
print(f"  Saved to: {OUTPUT_PATH}")

Copying accident dataset...
  [train] acc: 23808 images copied
  [val] acc: 2501 images copied
  [test] acc: 1245 images copied
Copying fire dataset...
  [train] fire: 14122 images copied
  [val] fire: 3099 images copied
  [test] fire: 4306 images copied

✅ Merge complete!
  train: 37930 images
  val: 5600 images
  test: 5551 images
  Total classes: 24
  Saved to: /kaggle/working/v2_acc_and_fire_datasets_merged


## ✅ Initializing the Run

In [5]:
# ── W&B: Initialize run with full hyperparameter config ──────────────
EPOCHS = 50
IMGSZ  = 640
BATCH  = 64
MODEL  = "yolo26n.pt"
PROJECT = "Fire_Plus_Accidents"
RUN_NAME = "v4_Fire_And_Accident_detection_AG"           # edit AG to your short name & start with v1 & edit the version number as you go !!!!

run = wandb.init(
    project=PROJECT,
    name=RUN_NAME,
    job_type="training",
    config = {
        "model":        MODEL,
        "pretrained":   True,
        "time":         11.0,
        
        # Core Training
        "epochs":          EPOCHS,
        "imgsz":           IMGSZ,
        "batch":           BATCH,
        
        # Optimizer & LR
        "optimizer"     : "MuSGD",    
        "lr0"           : 0.015342745276421436,
        "lrf"           : 0.018045285401621577,
        "momentum"      : 0.8512772813546273,       
        "weight_decay"  : 0.0004355163293740202,
        "warmup_epochs" : 2.245050203259325,

        # Training stability
        "patience": 15,               # early stopping
        "cos_lr": True,

        # Dataset
        "dataset_link":    ["https://www.kaggle.com/datasets/maryamsamirelsayed/accident-types-and-detection-dataset", "https://www.kaggle.com/datasets/sayedgamal99/smoke-fire-detection-yolo"],
        "num_classes":     24,
}
)
print(f"W&B run started: {run.url}")


W&B run started: https://wandb.ai/ahmed-hossam-suez-canal-university/Fire_Plus_Accidents/runs/dap3f5fx


## ✅ Modeling

In [6]:
from ultralytics import YOLO

cfg = wandb.config  # use values logged to W&B

model = YOLO(cfg.model)

results = model.train(
    data='/kaggle/working/v2_acc_and_fire_datasets_merged/data.yaml',

    time          = cfg.time,
    
    # Core Training
    epochs        = cfg.epochs,
    imgsz         = cfg.imgsz,
    batch         = cfg.batch,

    # Optimizer & LR 
    optimizer     = cfg.optimizer,    
    lr0           = cfg.lr0,
    lrf           = cfg.lrf,
    momentum      = cfg.momentum,       
    weight_decay  = cfg.weight_decay,
    warmup_epochs = cfg.warmup_epochs,

    # Training stability
    patience      = cfg.patience,   
    cos_lr        = cfg.cos_lr,   

    # Logging
    project       = PROJECT,
    name          = RUN_NAME,
    plots         = True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.60 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/v2_acc_and_fire_datasets_merged/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.

## ✅ Logging the Results

In [7]:
# ── W&B: Log final validation metrics ─────────────────────────────────
metrics_dict = results.results_dict

final_metrics = {
    "final/precision":    metrics_dict.get("metrics/precision(B)", 0),
    "final/recall":       metrics_dict.get("metrics/recall(B)",    0),
    "final/mAP50":        metrics_dict.get("metrics/mAP50(B)",     0),
    "final/mAP50-95":     metrics_dict.get("metrics/mAP50-95(B)",  0),
    "final/fitness":      results.fitness,
}
wandb.log(final_metrics)

# Also write them as W&B summary so they show in the runs table
for k, v in final_metrics.items():
    wandb.run.summary[k] = v

print("Logged metrics:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.4f}")


Logged metrics:
  final/precision: 0.7104
  final/recall: 0.6351
  final/mAP50: 0.7025
  final/mAP50-95: 0.4351
  final/fitness: 0.4351


In [8]:
# ── W&B: Log training plots & validation images ───────────────────────
from pathlib import Path

save_dir = Path(results.save_dir)

plot_files = {
    "confusion_matrix":           save_dir / "confusion_matrix.png",
    "confusion_matrix_normalized": save_dir / "confusion_matrix_normalized.png",
    "BoxPR_curve":                   save_dir / "BoxPR_curve.png",
    "BoxF1_curve":                   save_dir / "BoxF1_curve.png",
    "BoxP_curve":                    save_dir / "BoxP_curve.png",
    "BoxR_curve":                    save_dir / "BoxR_curve.png",
    "results":                    save_dir / "results.png",
    "labels":                     save_dir / "labels.jpg"
}

wandb_images = {}
for name, path in plot_files.items():
    if path.exists():
        wandb_images[f"plots/{name}"] = wandb.Image(str(path), caption=name)
        print(f"  ✓ {name}")
    else:
        print(f"  ✗ {name} not found")

# Validation batch predictions (ground-truth vs predictions)
for img_path in sorted(save_dir.glob("val_batch*.jpg")):
    wandb_images[f"val_batches/{img_path.stem}"] = wandb.Image(
        str(img_path), caption=img_path.stem
    )
    print(f"  ✓ {img_path.stem}")

wandb.log(wandb_images)
print(f"\nLogged {len(wandb_images)} images/plots to W&B.")


  ✓ confusion_matrix
  ✓ confusion_matrix_normalized
  ✓ BoxPR_curve
  ✓ BoxF1_curve
  ✓ BoxP_curve
  ✓ BoxR_curve
  ✓ results
  ✓ labels
  ✓ val_batch0_labels
  ✓ val_batch0_pred
  ✓ val_batch1_labels
  ✓ val_batch1_pred
  ✓ val_batch2_labels
  ✓ val_batch2_pred

Logged 14 images/plots to W&B.


In [9]:
# ── W&B: Save best model as a versioned artifact ──────────────────────
best_pt = save_dir / "weights" / "best.pt"
last_pt = save_dir / "weights" / "last.pt"

artifact = wandb.Artifact(
    name="Accident_and_fire_detector",
    type="model",
    description="YOLOv26n fine-tuned for Fire and Accident detection on merged dataset",
    metadata={
        "mAP50":     wandb.run.summary.get("final/mAP50"),
        "mAP50-95":  wandb.run.summary.get("final/mAP50-95"),
        "precision": wandb.run.summary.get("final/precision"),
        "recall":    wandb.run.summary.get("final/recall"),
        "epochs":    cfg.epochs,
        "imgsz":     cfg.imgsz,
        "dataset_link": cfg.dataset_link,
    }
)

if best_pt.exists():
    artifact.add_file(str(best_pt), name="best.pt")
if last_pt.exists():
    artifact.add_file(str(last_pt), name="last.pt")

wandb.log_artifact(artifact)
artifact.wait()  # ← block until artifact is fully logged on the W&B server
print(f"Model artifact logged: {artifact.name}:{artifact.version}")


Model artifact logged: Accident_and_fire_detector:v2:v2


In [10]:
# Finish the WandB run
wandb.finish()

final/fitness,▁
final/mAP50,▁
final/mAP50-95,▁
final/precision,▁
final/recall,▁
final/fitness,0.4351
final/mAP50,0.70251
final/mAP50-95,0.4351
final/precision,0.7104
final/recall,0.63505
